### Feature Engineering - Aviation Accident Modeling

This notebook prepares features for machine learning models to predict damage types. It performs comprehensive data preprocessing and feature engineering for aviation safety analysis and transforms raw accident data into machine learning-ready features for predictive modeling. 

The pipeline includes data cleaning, categorical encoding, numerical transformations, and feature selection. Output features can be used to build models for accident severity classification and damage assessment. The notebook also handles missing data imputation and creates derived features from temporal and categorical variables. 

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load cleaned data
df = pd.read_csv('data\CleanedAviationData.csv')
print(f"Dataset shape: {df.shape}")

Dataset shape: (88882, 22)


## Feature Selection and Engineering

In [4]:
# Selecting relevant features for modeling
feature_columns = [
    'Make', 'Model', 'Engine.Type',
    'Weather.Condition', 'Broad.phase.of.flight',
    'Purpose.of.flight', 'Amateur.Built', 'Month', 'Year'
]

# Target variables
target_columns = {
    # 'severity': 'Severity_Category',
    # 'fatal': 'Is_Fatal',
    'damage': 'Aircraft.damage'
}

# Create modeling dataset
model_df = df[feature_columns + list(target_columns.values())].copy()

# Remove rows with missing target values
model_df = model_df.dropna(subset=['Aircraft.damage'])

print(f"Modeling dataset shape: {model_df.shape}")
print(f"Missing values:\n{model_df.isnull().sum()}")

Modeling dataset shape: (88882, 10)
Missing values:
Make                     0
Model                    0
Engine.Type              0
Weather.Condition        0
Broad.phase.of.flight    0
Purpose.of.flight        0
Amateur.Built            0
Month                    0
Year                     0
Aircraft.damage          0
dtype: int64


## Categorical Encoding

In [ ]:
# Encode categorical variables
categorical_features = ['Make', 'Model', 'Engine.Type', 'Weather.Condition', 
                       'Broad.phase.of.flight', 'Purpose.of.flight', 'Amateur.Built']

# Label encode categorical features
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    model_df[col] = model_df[col].fillna('Unknown')
    model_df[col + '_encoded'] = le.fit_transform(model_df[col])
    label_encoders[col] = le

# Encode target variables
target_encoders = {}
for target_name, target_col in target_columns.items():
    if target_col != 'Is_Fatal':  # Already binary
        le = LabelEncoder()
        model_df[target_col + '_encoded'] = le.fit_transform(model_df[target_col])
        target_encoders[target_name] = le

print("Categorical encoding completed")

## Feature Matrix Creation

In [ ]:
# Create feature matrix
feature_cols_encoded = [col + '_encoded' for col in categorical_features] + \
                      ['Number.of.Engines', 'Month', 'Year']

X = model_df[feature_cols_encoded].fillna(0)

# Target variables
y_severity = model_df['Severity_Category_encoded']
y_fatal = model_df['Is_Fatal']
y_damage = model_df['Aircraft.damage_encoded']

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {feature_cols_encoded}")
print(f"\nTarget distributions:")
print(f"Severity: {model_df['Severity_Category'].value_counts()}")
print(f"Fatal: {model_df['Is_Fatal'].value_counts()}")
print(f"Damage: {model_df['Aircraft.damage'].value_counts()}")

## Train-Test Split

In [ ]:
# Split data for each target
test_size = 0.2
random_state = 42

# Severity prediction split
X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(
    X, y_severity, test_size=test_size, random_state=random_state, stratify=y_severity
)

# Fatal prediction split
X_train_fatal, X_test_fatal, y_train_fatal, y_test_fatal = train_test_split(
    X, y_fatal, test_size=test_size, random_state=random_state, stratify=y_fatal
)

# Damage prediction split
X_train_dmg, X_test_dmg, y_train_dmg, y_test_dmg = train_test_split(
    X, y_damage, test_size=test_size, random_state=random_state, stratify=y_damage
)

print(f"Training set sizes:")
print(f"Severity: {X_train_sev.shape[0]}")
print(f"Fatal: {X_train_fatal.shape[0]}")
print(f"Damage: {X_train_dmg.shape[0]}")

## Save Processed Data

In [ ]:
# Save feature-engineered data
model_df.to_csv('data/cleaned_data/aviation_features.csv', index=False)

# Save train-test splits
import pickle

splits_data = {
    'severity': (X_train_sev, X_test_sev, y_train_sev, y_test_sev),
    'fatal': (X_train_fatal, X_test_fatal, y_train_fatal, y_test_fatal),
    'damage': (X_train_dmg, X_test_dmg, y_train_dmg, y_test_dmg),
    'encoders': {'label_encoders': label_encoders, 'target_encoders': target_encoders},
    'feature_names': feature_cols_encoded
}

with open('data/cleaned_data/model_splits.pkl', 'wb') as f:
    pickle.dump(splits_data, f)

print("Feature engineering completed and saved!")
print(f"Features ready for modeling: {len(feature_cols_encoded)}")
print(f"Total samples: {len(X):,}")